# Tutorial 8b: Recovering a Policy from a Manifest

A fresh process has nothing but AWS credentials. In this tutorial you
will bootstrap a minimal policy with just an S3 pool, recover the full
environment manifest stored in **Tutorial 8a**, and replay it with a
single assignment: `laila.args.environment = recovered_env`.

That assignment triggers `_load_environment`, which calls
`laila.terminate(...)` to tear down the bootstrap policy and rebuilds
every policy listed in the snapshot — pools, taskforces, and
communication protocols included — then activates the one identified by
`active_gid`.

In this tutorial you will:

- Bootstrap a minimal policy with only S3 access
- Use `laila.remember` to recover the environment manifest
- Replay the snapshot via `laila.args.environment = ...`
- Verify the new active policy carries the full multi-pool setup

**Prerequisites:** `pip install "laila-core[s3]"`, a `secrets.toml` with AWS credentials, and a completed run of **Tutorial 8a** (the manifest must be in S3).

In [ ]:
import json
import laila
from laila.pool import S3Pool

## Step 1: Bootstrap a minimal policy

This is the "old" policy. It has a single S3 pool — just enough to
reach the manifest stored in Tutorial 8a. No HDF5, no extra
configuration. Accessing `laila.memory` lazily creates a `DefaultPolicy`,
so we just register the pool.

In [ ]:
laila.read_args("./secrets.toml")

bootstrap_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
    nickname="s3",
)
laila.memory.extend(bootstrap_pool, pool_nickname="s3")

print(f"Bootstrap policy: {laila.active_policy.global_id}")
print(f"Pools: {list(laila.active_policy.central.memory.pool_router.pools.keys())}")

## Step 2: Remember the environment manifest

Reconstruct the manifest identity from its nickname, remember the
blueprint, then ask the rebuilt manifest for its **realized** form to
fetch the leaf entry that holds the environment dict.

In [ ]:
manifest = laila.manifest(nickname="env_manifest_v1")

with laila.guarantee:
    ref = laila.remember(manifest.global_id, pool_nickname="s3")

blueprint = ref.data
manifest = laila.manifest(data=blueprint, nickname="env_manifest_v1")

print(f"Blueprint: {manifest.blueprint}")

In [ ]:
with laila.guarantee:
    manifest.remember(pool_nickname="s3")

recovered_env = manifest.realized["config"].data

print("Recovered environment:")
print(json.dumps(recovered_env, indent=2, default=str))

## Step 3: Replay the snapshot

Assigning the recovered dict to `laila.args.environment` triggers the
loader: it tears down the bootstrap policy with `laila.terminate(...)`,
rebuilds every policy listed under `policies` (with the right pool /
taskforce / protocol subclasses), and activates the one identified by
`active_gid`.

Because the snapshot in 8a captured the **full** environment plus the
active policy's `global_id`, no manual wiring is required here — one
assignment is enough.

In [ ]:
laila.args.environment = recovered_env

new_policy = laila.active_policy
print(f"New active policy: {new_policy.global_id}")
print(f"Pools on new policy: {list(new_policy.central.memory.pool_router.pools.keys())}")

## Step 4: Verify the new policy

The new policy should carry the full multi-pool setup from Tutorial 8a,
not just the single S3 pool the bootstrap policy had. The bootstrap
policy itself no longer exists — `laila.terminate(...)` ran inside
`_load_environment` and removed it from `laila.local_policies`.

In [ ]:
new_env = laila.args.environment.policies[laila.active_policy.global_id].toDict()

new_pools = new_env["central"]["memory"]["pool_router"]["pools"]
old_pools = (
    recovered_env["policies"][recovered_env["active_gid"]]
    ["central"]["memory"]["pool_router"]["pools"]
)

print(f"Recovered environment had {len(old_pools)} pool(s)")
print(f"New policy has {len(new_pools)} pool(s)")
print(f"Live local policies: {len(laila.local_policies)}")

for pool_id, pool_cfg in new_pools.items():
    print(f"\n  Pool: {pool_id}")
    for k, v in pool_cfg.items():
        print(f"    {k}: {v}")

## Step 5: Clean up

Remove the environment manifest and its leaf entry from S3. The new
policy already has the `"s3"` pool registered (it came from the
restored environment), so we can call `manifest.forget` directly.

In [ ]:
with laila.guarantee:
    manifest.forget(pool_nickname="s3")

print("Manifest and leaf entries cleaned up from S3")

## Summary

- Only S3 credentials were needed to recover the full policy configuration
- `laila.remember` fetched the manifest blueprint, then `manifest.realized` materialised the environment dict
- **Assigning `laila.args.environment = recovered_env` triggered `_load_environment`**, which:
  1. tore down the bootstrap policy via `laila.terminate(...)`,
  2. rebuilt every policy listed in the snapshot — pools, taskforces, protocols included,
  3. activated the one identified by `active_gid`.
- This pattern enables portable policy snapshots, migration between environments, and disaster recovery